In [1]:
!pip install transformers torch scikit-learn pandas

In [2]:
!pip install mlflow


In [3]:
import mlflow

# Create or connect to a specific experiment for your notebook explorations
mlflow.set_experiment("reviewpulse-notebook-exploration")

<Experiment: artifact_location='/kaggle/working/mlruns/1', creation_time=1793889187228, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1793889187228, lifecycle_stage='active', name='reviewpulse-notebook-exploration', tags={}, trace_location=None, workspace='default'>

In [4]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score
import mlflow

# 1. Load validation data (adjust path if needed for Kaggle)
df_val = pd.read_csv("/kaggle/input/datasets/muskaanadwani/reviewpulse-validation/validation.csv")
texts = df_val["review_text"].tolist()
y_true = df_val["label"].tolist()

# 2. Define explicit mapping policies
def map_stars_to_binary(label: str) -> int:
    """
    Policy for nlptown (1 to 5 stars).
    1-2 stars -> negative (0). 4-5 stars -> positive (1).
    3 stars are explicitly mapped to negative (0) to enforce strict quality.
    """
    if label in ["1 star", "2 stars", "3 stars"]:
        return 0
    elif label in ["4 stars", "5 stars"]:
        return 1
    else:
        raise ValueError(f"Unexpected label: {label}")

def map_binary(label: str) -> int:
    """Policy for standard binary English models (SST-2)."""
    return 1 if label.upper() == "POSITIVE" else 0

def map_roberta_3class(label: str) -> int:
    """
    Policy for 3-class RoBERTa models.
    LABEL_0 (negative) -> 0. LABEL_2 (positive) -> 1.
    LABEL_1 (neutral) is explicitly mapped to negative (0).
    """
    if label.upper() in ["LABEL_0", "NEGATIVE"]:
        return 0
    elif label.upper() in ["LABEL_2", "POSITIVE"]:
        return 1
    elif label.upper() in ["LABEL_1", "NEUTRAL"]:
        return 0  # Explicit policy for neutral
    else:
        raise ValueError(f"Unexpected label: {label}")

# 3. Initialize candidate models (device=0 uses the Kaggle GPU T4)
candidates = {
    "nlptown_1_to_5": {
        "repo": "nlptown/bert-base-multilingual-uncased-sentiment",
        "mapper": map_stars_to_binary
    },
    "distilbert_sst2": {
        "repo": "distilbert-base-uncased-finetuned-sst-2-english",
        "mapper": map_binary
    },
    "roberta_sentiment": {
        # Using a popular RoBERTa sentiment checkpoint
        "repo": "cardiffnlp/twitter-roberta-base-sentiment", 
        "mapper": map_roberta_3class
    }
}

# 4. Set the MLflow Experiment
mlflow.set_experiment("reviewpulse-bert-candidates")
results = []

# 5. Run inference and log to MLflow
for name, config in candidates.items():
    print(f"\nEvaluating {name} ({config['repo']})...")
    
    # Start an MLflow run for this specific model
    with mlflow.start_run(run_name=name):
        
        # Log model configurations
        mlflow.log_param("repo", config["repo"])
        mlflow.log_param("mapper_policy", config["mapper"].__name__)
        mlflow.log_param("max_length", 512)
        mlflow.log_param("truncation", True)
        
        # Load pipeline with truncation to handle long Amazon reviews
        classifier = pipeline(
            "sentiment-analysis", 
            model=config["repo"], 
            device=0, 
            truncation=True, 
            max_length=512
        )
        
        # Run predictions in batches for speed on GPU
        raw_preds = classifier(texts, batch_size=32)
        
        # Apply the specific label mapping policy for this model
        y_pred = [config["mapper"](p["label"]) for p in raw_preds]
        
        # Calculate macro-F1 (primary) and accuracy (secondary)
        acc = accuracy_score(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average="macro")
        
        # Log results to MLflow
        mlflow.log_metric("val_accuracy", acc)
        mlflow.log_metric("val_macro_f1", macro_f1)
        
        results.append({
            "Model": name,
            "Accuracy": round(acc, 4),
            "Macro-F1": round(macro_f1, 4)
        })

# 6. Display results
print("\n--- Model Comparison Results ---")
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


Evaluating nlptown_1_to_5 (nlptown/bert-base-multilingual-uncased-sentiment)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Evaluating distilbert_sst2 (distilbert-base-uncased-finetuned-sst-2-english)...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Evaluating roberta_sentiment (cardiffnlp/twitter-roberta-base-sentiment)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- Model Comparison Results ---
            Model  Accuracy  Macro-F1
   nlptown_1_to_5     0.930    0.9298
  distilbert_sst2     0.891    0.8909
roberta_sentiment     0.897    0.8969


In [5]:
!zip -r mlflow_archive.zip /kaggle/working/mlruns /kaggle/working/*.db

	zip warning: name not matched: /kaggle/working/mlruns
  adding: kaggle/working/mlflow.db (deflated 98%)
